In [1]:
# imports

from armor_tools import analysis
from armor_tools import plot
from pathlib import Path
from datetime import datetime, timedelta
import pyart
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np 
import matplotlib.pyplot as plt



## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [30]:
ROOT = Path('/nas/rhome/eebbert/armor/cfrad/')

# (year, month, date, hour, minute, second)
start_datetime = datetime(2026, 6, 22, 12, 0, 0)
end_datetime   = datetime(2026, 6, 23, 5, 0, 0)

#range for plots
rng = 75

In [31]:
# finding folders in timerange
print(f'User defined folder: {ROOT}')
print(f'Searching for files between {start_datetime} and {end_datetime} ...')
files = analysis.find_files_in_timerange(ROOT, start_datetime, end_datetime)
print(f'Found {len(files)} matching files.')

# filtering to only include sector scans
ppi_sector_files = analysis.filter_files_vcp(files, vcp_min=300, vcp_max=400)
print(f'PPI Sector Files: {len(ppi_sector_files)}')

User defined folder: /nas/rhome/eebbert/armor/cfrad
Searching for files between 2026-06-22 12:00:00 and 2026-06-23 05:00:00 ...
Found 94 matching files.
PPI Sector Files: 16


In [32]:
# function to plot a widget that can loop through elevations in a scan
def plot_ppi_sector_widget(radar, fields, xmin=-rng, xmax=rng, ymin=-rng, ymax=rng, grids=False):
    """
    Interactive widget to browse sweeps in a PPI sector scan.

    Parameters
    ----------
    radar : pyart.core.Radar
        PyART radar object (sector scan).
    fields : list of str
        Field names to plot side-by-side (e.g. ['REF', 'VEL']).
    xmin, xmax : float
        East-West plot extent in km. Defaults to -30 to 30.
    ymin, ymax : float
        North-South plot extent in km. Defaults to -30 to 30.
    grids : bool
        If True, draw major and minor gridlines.
    """

    FIELD_PARAMS = {
        'reflectivity':             {'vmin': -10, 'vmax': 70,   'cmap': 'HomeyerRainbow',  'title': 'Reflectivity (dBZ)'},
        'REF':                      {'vmin': -10, 'vmax': 70,   'cmap': 'HomeyerRainbow',  'title': 'Reflectivity (dBZ)'},
        'FREF':                     {'vmin': -10, 'vmax': 70,   'cmap': 'HomeyerRainbow',  'title': 'Reflectivity (dBZ)'},
        'corrected_reflectivity':   {'vmin': -10, 'vmax': 70,   'cmap': 'HomeyerRainbow',  'title': 'Reflectivity (dBZ)'},
        'velocity':                 {'vmin': -16, 'vmax': 16,   'cmap': 'PuOr_r',          'title': 'Radial Velocity (m/s)'},
        'VEL':                      {'vmin': -16, 'vmax': 16,   'cmap': 'PuOr_r',          'title': 'Radial Velocity (m/s)'},
        'FVEL':                     {'vmin': -16, 'vmax': 16,   'cmap': 'PuOr_r',          'title': 'Radial Velocity (m/s)'},
        'differential_reflectivity':{'vmin': -2,  'vmax': 6,    'cmap': 'ChaseSpectral',   'title': 'Differential Reflectivity (dB)'},
        'ZDR':                      {'vmin': -2,  'vmax': 6,    'cmap': 'ChaseSpectral',   'title': 'Differential Reflectivity (dB)'},
        'FZDR':                     {'vmin': -2,  'vmax': 6,    'cmap': 'ChaseSpectral',   'title': 'Differential Reflectivity (dB)'},
        'cross_correlation_ratio':  {'vmin': 0.4, 'vmax': 1.05, 'cmap': 'plasma',          'title': 'RHO (ρhv)'},
        'RHO':                      {'vmin': 0.4, 'vmax': 1.05, 'cmap': 'plasma',          'title': 'RHO (ρhv)'},
        'FRHO':                     {'vmin': 0.4, 'vmax': 1.05, 'cmap': 'plasma',          'title': 'RHO (ρhv)'},
        'spectrum_width':           {'vmin': 0,   'vmax': 10,   'cmap': 'pyart_NWS_SPW',   'title': 'Spectrum Width (m/s)'},
        'SW':                       {'vmin': 0,   'vmax': 10,   'cmap': 'pyart_NWS_SPW',   'title': 'Spectrum Width (m/s)'},
        'FSW':                      {'vmin': 0,   'vmax': 10,   'cmap': 'pyart_NWS_SPW',   'title': 'Spectrum Width (m/s)'},
        'differential_phase':       {'vmin': 0,   'vmax': 180,  'cmap': 'viridis',         'title': 'Differential Phase (deg)'},
        'PHI':                      {'vmin': 0,   'vmax': 180,  'cmap': 'viridis',         'title': 'Differential Phase (deg)'},
        'FPHI':                     {'vmin': 0,   'vmax': 180,  'cmap': 'viridis',         'title': 'Differential Phase (deg)'},
    }

    units_str = radar.time['units']
    base_time_str = units_str.split('since')[-1].strip().replace('Z', '')
    base_time = datetime.fromisoformat(base_time_str)
    radar_display = pyart.graph.RadarDisplay(radar)

    out = widgets.Output()

    def _plot(sweep_num):
        out.clear_output(wait=True)
        with out:
            sweep_start = radar.sweep_start_ray_index['data'][sweep_num]
            elevation = radar.elevation['data'][sweep_start]
            sweep_time_seconds = round(radar.time['data'][sweep_start])
            sweep_datetime = base_time + timedelta(seconds=float(sweep_time_seconds))
            sweep_time_str = f"{sweep_datetime.isoformat()}Z"

            nplots = len(fields)
            fig, axes = plt.subplots(1, nplots, figsize=(6 * nplots, 5))
            axes = np.atleast_1d(axes).flatten()

            for i, field in enumerate(fields):
                params = FIELD_PARAMS.get(
                    field,
                    {'vmin': None, 'vmax': None, 'cmap': 'viridis', 'title': field}
                )
                ax = axes[i]
                radar_display.plot_ppi(
                    field, sweep=sweep_num, ax=ax,
                    vmin=params['vmin'], vmax=params['vmax'],
                    cmap=params['cmap'],
                    colorbar_label=params['title'],
                    title=f"ARMOR PPI Sector | {params['title']}\nEl: {elevation:.1f}°  |  {sweep_time_str}"
                )
                ax.set_xlim(xmin, xmax)
                ax.set_ylim(ymin, ymax)
                ax.set_xlabel('Distance from Radar (E/W) (km)')
                ax.set_ylabel('Distance from Radar (N/S) (km)')

                if grids:
                    ax.minorticks_on()
                    ax.grid(which='major', color='gray', linestyle='-', linewidth=0.8)
                    ax.grid(which='minor', color='gray', linestyle='--', linewidth=0.5)

            plt.tight_layout()
            plt.show()

    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=radar.nsweeps - 1,
        step=1,
        description=f'Sweep (0–{radar.nsweeps - 1}):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px'),
        continuous_update=False
    )

    # show elevation label next to slider
    elev_label = widgets.Label()

    def _update_label(change):
        snum = change['new']
        s = radar.sweep_start_ray_index['data'][snum]
        elev = radar.elevation['data'][s]
        elev_label.value = f"El: {elev:.2f}°"

    slider.observe(_update_label, names='value')
    _update_label({'new': 0})

    widgets.interactive_output(_plot, {'sweep_num': slider})
    display(widgets.VBox([widgets.HBox([slider, elev_label]), out]))
    _plot(0)

In [ ]:
# selecting the first sector file as a test
test_file = ppi_sector_files[2]

f_nc = analysis.decompress_xz(test_file)

radar = pyart.io.read(f_nc)


plot_ppi_sector_widget(radar=radar, fields=['REF'], xmin=42, xmax=62, ymin=-10, ymax=10, grids=True)



In [ ]:
cradar = analysis.correct_azimuth_pointing_angle_sector_dynamic(radar)
plot_ppi_sector_widget(radar=cradar, fields=['REF'], xmin=42, xmax=62, ymin=-10, ymax=10, grids=True)
